# 08 - Morphotope-Level Characters

Following source `07_morphotope_chars.ipynb`:
Compute morphotope-level geometric features that go beyond building aggregation:
- `limLPS` — linear plan structure indicator
- `limAre/limPer/limLAL` — sums of top-10 connected building groups
- `limSDi` — spacing regularity via relative-neighborhood triangulation
- `limMFR/limMTC/limLMFR` — facade ratios and thin building ratios

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else ".")

import numpy as np
import pandas as pd
import geopandas as gpd
import momepy as mm
from libpysal.graph import Graph, read_parquet
from shapely import unary_union
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

MORPH_DIR = Path("outputs/06_morphotopes")
OUT = Path("outputs/08_morphotope_chars")
OUT.mkdir(parents=True, exist_ok=True)

c:\urban-form\urban_taxonomy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tessellation = gpd.read_parquet("outputs/02_tessellation/tessellation.parquet")
buildings    = gpd.read_file("outputs/01_preprocessed/buildings.gpkg")
bq1          = read_parquet("outputs/03_graphs/building_graph.parquet")
labels       = pd.read_parquet(MORPH_DIR / "tessellation_labels_postprocessed.parquet")["morphotope_label"]

# Map tessellation cells to morphotope labels
tessellation["morph"] = labels
print(f"{labels.nunique()} morphotopes")

1996 morphotopes


In [3]:
LIMIT_VALUE = 10  # top N connected building groups

def generate_ratio(group_idx, buildings, bq1):
    """Compute morphotope-level chars for a group of buildings (source logic)."""
    grp_buildings = buildings.loc[group_idx]
    if len(grp_buildings) < 3:
        return pd.Series({
            'limMFR': np.nan, 'limMTC': np.nan, 'limLMFR': np.nan,
            'limLPS': 0, 'limAre': grp_buildings.area.sum(),
            'limPer': grp_buildings.length.sum(),
            'limLAL': 0, 'limSDi': 0.35,
        })

    sub_bq1 = bq1.subgraph(group_idx)
    comp_labels = sub_bq1.component_labels

    # Union connected building groups
    connected = grp_buildings.geometry.groupby(comp_labels).apply(
        lambda x: unary_union(x.values)
    )
    connected = gpd.GeoSeries(connected, crs=buildings.crs).buffer(0.1).normalize().make_valid()

    fr = mm.facade_ratio(connected)
    elo = mm.elongation(connected)

    # "Thin" buildings: low facade ratio AND not too elongated
    thin = (fr < 8) & (elo < 0.90)
    areas = connected.area
    morph_fr_area_ratio = areas[thin].sum() / areas.sum() if areas.sum() > 0 else 0

    libNCos = grp_buildings["libNCo"].sum() if "libNCo" in grp_buildings.columns else 0
    limLPS = 1 if (morph_fr_area_ratio > 0.4 or libNCos > 0) else 0

    # Top-N sums
    top = min(LIMIT_VALUE, len(connected))
    sorted_areas = connected.area.sort_values(ascending=False)
    limAre = sorted_areas.iloc[:top].sum()
    sorted_pers = connected.length.sort_values(ascending=False)
    limPer = sorted_pers.iloc[:top].sum()
    lals = mm.longest_axis_length(connected)
    sorted_lals = lals.sort_values(ascending=False)
    limLAL = sorted_lals.iloc[:top].sum()

    # Spacing regularity
    try:
        rn = Graph.build_triangulation(
            grp_buildings.representative_point(), method="relative_neighborhood", kernel="identity"
        )
        dists = rn.aggregate("median")
        limSDi = dists.std() / grp_buildings.geometry.length.median() if grp_buildings.geometry.length.median() > 0 else 0
    except Exception:
        limSDi = 0.35

    return pd.Series({
        'limMFR': fr.median(), 'limMTC': thin.sum() / len(thin) if len(thin) > 0 else 0,
        'limLMFR': morph_fr_area_ratio, 'limLPS': limLPS,
        'limAre': limAre, 'limPer': limPer, 'limLAL': limLAL, 'limSDi': limSDi,
    })


# Group buildings by morphotope and compute chars
bldg_morph = buildings.merge(
    tessellation[["morph"]], left_index=True, right_index=True, how="inner"
)

morph_chars = bldg_morph.groupby("morph").apply(
    lambda g: generate_ratio(g.index, buildings, bq1), include_groups=False
)
print(f"Morphotope chars: {morph_chars.shape}")
print(morph_chars.describe().round(2))

Morphotope chars: (1996, 8)
        limMFR   limMTC  limLMFR   limLPS     limAre   limPer   limLAL  \
count  1890.00  1890.00  1890.00  1996.00    1996.00  1996.00  1996.00   
mean      2.72     0.78     0.74     0.90    7021.97   964.89   327.14   
std       0.91     0.12     0.18     0.31   16755.09   856.77   286.34   
min       1.46     0.00     0.00     0.00       1.04     4.09     0.00   
25%       2.21     0.71     0.65     1.00    1746.20   581.95   202.66   
50%       2.57     0.78     0.76     1.00    2495.97   724.60   249.43   
75%       2.95     0.86     0.85     1.00    5026.86  1008.96   340.91   
max      11.86     1.00     1.00     1.00  266465.74  8158.57  2676.63   

        limSDi  
count  1996.00  
mean      0.25  
std       0.18  
min       0.01  
25%       0.15  
50%       0.20  
75%       0.30  
max       2.23  


In [4]:
morph_chars.to_parquet(OUT / "morph_chars.parquet")
print(f"Saved to {OUT / 'morph_chars.parquet'}")

Saved to outputs\08_morphotope_chars\morph_chars.parquet
